# SAURIA + Gather full environment notebook

Questo notebook crea una workload SAURIA, genera gli stimuli vanilla tramite `sauria_lib.py`, poi inserisce automaticamente gli stimuli gather in modalità **DRAM writeback**:

```text
gather legge dense/CSR/idx da DRAM esterna
gather scrive IFMAP gathered in DRAM esterna
SAURIA parte normalmente e carica IFMAP con il suo DMA standard
```

Questa versione non modifica `gather_frontend_axi`; richiede però che in `sauria_subsystem.sv` i canali gather siano instradati così durante `gather_busy`:

```text
gather_ext_mem.AR/R       -> DRAM esterna
gather_sauria_mem.AW/W/B  -> DRAM esterna
dma_ext_mem               -> DRAM esterna quando gather_busy=0
```


In [1]:
from pathlib import Path
import shutil
import subprocess
import sys
import os
import numpy as np
import torch
import dotenv

# Adjust these if your notebook is not run from Python/notebooks.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SAURIA_ROOT = REPO_ROOT  # Backward-compatible alias used by some debug cells.
PYTHON_DIR = REPO_ROOT / "Python"
TEST_DIR = REPO_ROOT / "test"
VERILATOR_DIR = TEST_DIR / "verilator"
STIMULI_DIR = TEST_DIR / "stimuli"
VANILLA_STIMULI_DIR = TEST_DIR / "stimuli_vanilla_for_gather"
GATHER_STIMULI_DIR = TEST_DIR / "stimuli"

dotenv.load_dotenv(REPO_ROOT / "env", override=True)
sys.path.insert(1, str(PYTHON_DIR))
# Put sauria_gather_env.py in Python/src or in the same folder as this notebook.
sys.path.insert(1, str(NOTEBOOK_DIR))
sys.path.insert(1, str(PYTHON_DIR / "src"))

import src.hw_versions as hwv
import src.sauria_lib as slib
import sauria_gather_env as genv

print("REPO_ROOT:", REPO_ROOT)
print("STIMULI_DIR:", STIMULI_DIR)


REPO_ROOT: /home/henry/sauria
STIMULI_DIR: /home/henry/sauria/test/stimuli


## 1. Hardware version

Usa la stessa versione che hai compilato in Verilator.


In [2]:
sauria_version = "FP16_8x16_AXI64"
HW_PARAMS = hwv.get_params(sauria_version)
print(HW_PARAMS)


{'MainMemory_offset': 0, 'SAURIA_offset_DMA': 3489660928, 'CTRL_offset': 1073741824, 'CORE_offset': 1342177280, 'DMA_offset': 1610612736, 'CFG_CON_offset': 512, 'CFG_IFM_offset': 1024, 'CFG_WEI_offset': 1536, 'CFG_PSM_offset': 2048, 'MEMA_offset': 262144, 'MEMB_offset': 524288, 'MEMC_offset': 786432, 'CFG_AXI_DATA_WIDTH': 32, 'CFG_AXI_ADDR_WIDTH': 32, 'MEMA_DEPTH': 2048, 'MEMB_DEPTH': 1024, 'MEMC_DEPTH': 2048, 'DATA_AXI_DATA_WIDTH': 64, 'DATA_AXI_ADDR_WIDTH': 32, 'X': 16, 'Y': 8, 'DILP_W': 64, 'PARAMS_W': 8, 'TH_W': 2, 'IFM_FIFO_POSITIONS': 5, 'WEI_FIFO_POSITIONS': 4, 'FIFO_FILL_CYCLES': 1, 'IA_W': 16, 'IB_W': 16, 'OC_W': 16, 'OP_TYPE': 1, 'IA_MANT': 10, 'IB_MANT': 10, 'IC_MANT': 10, 'rounding': 'RNE', 'approx_comp': False, 'mul_type': 3, 'M': 14, 'add_type': 4, 'A': 16, 'MEMA_W': 128, 'MEMB_W': 256, 'MEMC_W': 128, 'ADRA_W': 11, 'ADRB_W': 10, 'ADRC_W': 11, 'MEMA_N': 8, 'IFM_WOFS_W': 3, 'IFM_IDX_W': 15, 'MEMB_N': 16, 'WEI_WOFS_W': 4, 'WEI_IDX_W': 15, 'MEMC_N': 8, 'PSM_WOFS_W': 3, 'PSM_I

## 2. Workload minima legale per debug gather

Per `FP16_8x16`, `get_conv_dict` richiede:

```text
Cw_tile % Y_used == 0
C_out_tile % X_used == 0
```

Quindi il caso piccolo sotto usa `C_out_tile=16` e `Cw_tile=8`.


In [3]:
# Minimal legal smoke-test convolution.
C_in = 16
C_out = 16
Kh, Kw = 1, 1
s = 1
d = 1

B_conv_torch = torch.nn.Conv2d(C_in, C_out, (Kh, Kw), stride=s, dilation=d, bias=False)

Cw = 8
Ch = 1
Aw = (1 + s * (Cw - 1)) + (1 + d * (Kw - 1)) - 1
Ah = (1 + s * (Ch - 1)) + (1 + d * (Kh - 1)) - 1

tensor_A_torch = torch.randn(C_in, Ah, Aw, dtype=torch.float32)
tensor_C_torch = B_conv_torch(tensor_A_torch)

tensor_A = np.array(tensor_A_torch.detach())
tensor_B = np.array(B_conv_torch.weight.detach())
preload_C = np.zeros([C_out, Ch, Cw])
tensor_C = np.array(tensor_C_torch.detach())

print("A:", tensor_A.shape)
print("B:", tensor_B.shape)
print("C:", tensor_C.shape)
print("IFMAP values:", C_in * Ah * Aw)


A: (16, 1, 8)
B: (16, 16, 1, 1)
C: (16, 1, 8)
IFMAP values: 128


/tmp/ipykernel_9265/1263994853.py:18: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  tensor_A = np.array(tensor_A_torch.detach())
/tmp/ipykernel_9265/1263994853.py:19: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  tensor_B = np.array(B_conv_torch.weight.detach())
/tmp/ipykernel_9265/1263994853.py:21: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn

In [4]:
tensor_shapes = [tensor_A.shape, tensor_B.shape, tensor_C.shape]

TILING_DICT = {
    "C_tile_shape": [16, 1, 8],   # [C_out_tile, Ch_tile, Cw_tile]
    "tile_cin": 16,
    "X_used": 16,
    "Y_used": 8,
}

CONV_DICT = slib.get_conv_dict(tensor_shapes, TILING_DICT, HW_PARAMS, d=d, s=s, preloads=False)
print(CONV_DICT)


{'B_w': 1, 'B_h': 1, 'C_w': 8, 'C_h': 1, 'C_c': 16, 'A_w': 8, 'A_h': 1, 'A_c': 16, 'AB_c': 16, 'd': 1, 's': 1, 'w_til': 8, 'h_til': 1, 'c_til': 16, 'k_til': 16, 'A_w_til': 8, 'A_h_til': 1, 'X_ext_tiles': 1, 'Y_ext_tiles': 1, 'K_ext_tiles': 1, 'C_ext_tiles': 1, 'N_total_tiles': 1, 'B_w_eff': 1, 'B_h_eff': 1, 'X_int_tiles': 1, 'Y_int_tiles': 1, 'K_int_tiles': 1, 'N_cswitch': 1, 'X_used': 16, 'Y_used': 8, 'preload_en': False, 'Dil_pat': 9223372036854775808, 'rows_active': 255, 'cols_active': 65535, 'lwoffs': array([0, 1, 2, 3, 4, 5, 6, 7]), 'thres': 0}


## 3. Genera stimuli vanilla SAURIA

Questa cella usa `sauria_lib.py` come nel notebook originale. Prima verifica che il test vanilla passi.

`generate_vcd=False` evita VCD enormi.


In [5]:
SAURIA_outputs, SAURIA_stats = slib.Conv2d_SAURIA(
    tensor_A,
    tensor_B,
    preload_C,
    tensor_C,
    CONV_DICT,
    HW_PARAMS,
    generate_vcd=False,
    print_statistics=True,
    silent=False,
)

# Salva una copia degli stimuli vanilla appena generati.
VANILLA_STIMULI_DIR.mkdir(parents=True, exist_ok=True)
genv.copy_stimuli_dir(STIMULI_DIR, VANILLA_STIMULI_DIR)
print("Copied vanilla stimuli to", VANILLA_STIMULI_DIR)



              TEST PASSED

****************************************
          SAURIA STATISTICS
****************************************
Total cycles:				636
Total operations:			4096
Average Throughput:			6.44 OP/cycle (2.52 %)

Number of tiles:			1
Core stall cycles:			102 (16.04 %)
Memory/CGF stall cycles:		514 (80.82 %)

SAURIA memory capacity (A|B|C):		32.0 | 32.0 | 32.0 [kB]
Utilized memory:			0.25 | 0.5 | 0.25 [kB] (0.78 | 1.56 | 0.78 [%])
Copied vanilla stimuli to /home/henry/sauria/test/stimuli_vanilla_for_gather


## 4. Scegli gli indirizzi DRAM IFMAP per il gather

Per il mini-test generato sopra, SAURIA ha `Number of tiles: 1`, quindi usa un solo indirizzo DRAM IFMAP.

Non usare automaticamente un vecchio `sauria_debug.log`: se il log è della workload grande, il notebook genererebbe centinaia di gather e potrebbe anche sovrascrivere la metadata.


In [6]:
# Valori del tile IFMAP della workload minimal generata dal notebook.
N_VALUES = C_in * Ah * Aw          # 16 * 1 * 8 = 128
N_DENSE_COLS = N_VALUES
TILE_BYTES = N_VALUES * 2          # FP16 = 2 byte

# Mini-test: un solo tile IFMAP, letto da DRAM base 0x0 nella configurazione standard.
tile_out_bases = [0x0]

print("N_VALUES:", N_VALUES)
print("TILE_BYTES:", TILE_BYTES)
print("tile_out_bases:", [hex(x) for x in tile_out_bases])

# Solo se vuoi usare un trace nuovo e sei sicuro che sia della workload minimal, abilita questo blocco.
USE_TRACE_FOR_TILE_ADDRS = False
DEBUG_LOG = VERILATOR_DIR / "sauria_debug_minimal.log"

if USE_TRACE_FOR_TILE_ADDRS:
    tiles = genv.extract_ifmap_logical_tiles(
        DEBUG_LOG,
        ifmap_region_start=0x0,
        ifmap_region_end=0x10000,
        tile_bytes=TILE_BYTES,
    )
    genv.write_ifmap_tiles_csv(TEST_DIR / "ifmap_tiles_from_log.csv", tiles)
    tile_out_bases = sorted({t["dram_start_addr"] for t in tiles})
    print("Logical IFMAP reads:", len(tiles))
    print("Unique DRAM tile addresses:", len(tile_out_bases))
    print("First addresses:", [hex(x) for x in tile_out_bases[:8]])


N_VALUES: 128
TILE_BYTES: 256
tile_out_bases: ['0x0']


In [7]:
# Uses genv imported in the setup cell.
RESTORE_CHECK_DIR = REPO_ROOT / "test" / "stimuli_gather_restore_check"

manifest = genv.make_gather_restore_check_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=RESTORE_CHECK_DIR,
    tile_out_base=0x0,
    n_values=128,
    n_dense_cols=128,
    dense_stage_base=0x90000,
    poison_out=True,
)

print(manifest)


{'mode': 'gather_restore_check_only', 'tile_out_base': '0x00000000', 'n_values': 128, 'n_dense_cols': 128, 'check_start': '0x00000000', 'check_end_excl': '0x00000100', 'dense_stage_base': '0x00090000', 'comp_base': '0x00091000', 'idx_base': '0x00092000', 'metadata': {'comp_total_beats': 2, 'idx_total_beats': 65, 'idx_natural_beats': 64}, 'poison_out': True}


## 5. Genera stimuli SAURIA + gather

Questa cella prende gli stimuli vanilla e crea una nuova `test/stimuli` con il gather preposto a SAURIA.

Per la workload minima sopra, l’IFMAP teorico è `128` FP16. Però se dal trace vedi che SAURIA legge tile da `680` byte, usa `n_values=340` e gli indirizzi estratti dal log.


In [8]:
# Generate gather+SAURIA stimuli.
# Do not pass fixed comp_base/idx_base: the v2 helper places metadata after the dense staging area
# and checks for overlap automatically.
manifest = genv.make_gather_dram_writeback_stimuli(
    in_dir=VANILLA_STIMULI_DIR,
    out_dir=GATHER_STIMULI_DIR,
    tile_out_bases=tile_out_bases,
    n_values=N_VALUES,
    n_dense_cols=N_DENSE_COLS,
    dense_stage_base=0x90000,
    dense_stage_stride=0x1000,
    idx_extra_beats=1,
    poison_out=True,
    clear_keep_a_bit=True,
    insert_position="prepend",
    deduplicate_tiles=True,
)

print("Generated gather stimuli:")
print(manifest)


Generated gather stimuli:
{'mode': 'gather_dram_writeback_then_sauria_dma', 'num_unique_gather_tiles': 1, 'n_values': 128, 'n_dense_cols': 128, 'dense_stage_base': 589824, 'dense_stage_stride': 4096, 'comp_base': 593920, 'idx_base': 598016, 'idx_extra_beats': 1, 'metadata': {'comp_total_beats': 2, 'idx_total_beats': 65, 'idx_natural_beats': 64}, 'poison_out': True, 'poison_value': 0, 'clear_keep_a_bit': True, 'insert_position': 'prepend', 'input_dir': '/home/henry/sauria/test/stimuli_vanilla_for_gather', 'output_dir': '/home/henry/sauria/test/stimuli', 'tile_out_bases': ['0x00000000']}


## 6. Run Verilator su gather + SAURIA

Esegui senza VCD per il primo controllo. Il risultato atteso è:

```text
Benchmark passed with no errors.
SUCCESS!
```


In [9]:
cmd = ["./Test-Sim", "+debug", "+max-cycles=20000000"]
print("Running:", " ".join(cmd))
res = subprocess.run(cmd, cwd=VERILATOR_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(res.stdout[-5000:])
print("returncode:", res.returncode)


Running: ./Test-Sim +debug +max-cycles=20000000
890] CFG idx 32
[8900] CFG idx 33
Writing 0 into address 40000058
[8910] CFG idx 33
[8920] CFG idx 34
Writing 100 into address 4000005C
[8930] CFG idx 34
[8940] CFG idx 35
Writing 300 into address 40000060
[8950] CFG idx 35
[8960] CFG idx 36
Writing 83800000 into address 40000064
[8970] CFG idx 36
[8980] CFG idx 37
Writing 4000800F into address 40000068
[8990] CFG idx 37
[9000] CFG idx 38
Writing 0 into address 4000006C
[9010] CFG idx 38
[9020] CFG idx 39
Writing 40008 into address 40000070
[9030] CFG idx 39
[9040] CFG idx 40
Writing 10002 into address 40000074
[9050] CFG idx 40
[9060] CFG idx 41
Writing 20004008 into address 40000078
[9070] CFG idx 41
[9080] CFG idx 42
Writing 8001000 into address 4000007C
[9090] CFG idx 42
[9100] CFG idx 43
Writing 400 into address 40000080
[9110] CFG idx 43
[9120] CFG idx 44
Writing 0 into address 40000084
[9130] CFG idx 44
[9140] CFG idx 45
Writing 3FE00000 into address 40000088
[9150] CFG idx 45
[916

## 7. Se vuoi integrare in `sauria_lib.py`

La soluzione pulita è aggiungere a `sauria_lib.py` un wrapper opzionale che, dopo aver generato gli stimuli vanilla e prima di lanciare Verilator, chiama:

```python
genv.make_gather_dram_writeback_stimuli(...)
```

Per ora questo notebook evita modifiche invasive a `sauria_lib.py` e mantiene separati:

```text
SAURIA vanilla generation
Gather patch generation
Verilator run
```
